# Servidor Unificado en RunPod: Inferencia (M1 + M2 + M3) + Agente Ollama (`qwen3:1.7b`)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Levanta en un mismo Pod de RunPod (puerto HTTP **8000**):
- **M1:** Detección de placas + EasyOCR
- **M2:** Segmentación de instancias de vehículos (`YOLO11-seg`)
- **M3:** Keypoints sobre las ruedas (`YOLO11-pose`) $\rightarrow$ conteo de ejes
- **Agente LLM:** **Ollama (`qwen3:1.7b`)** local en el Pod (cumpliendo la Sección 4.8: *"con un LLM propio, por ejemplo, Ollama en RunPod"*).

In [ ]:
# 1. Configuración y URL pública del proxy de RunPod
import os, secrets

PUERTO = 8000
OLLAMA_MODEL = 'qwen3:4b'
API_TOKEN = os.environ.get('API_TOKEN') or secrets.token_urlsafe(16)
os.environ['API_TOKEN'] = API_TOKEN
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL

POD_ID = os.environ.get('RUNPOD_POD_ID')
URL_PUBLICA = f'https://{POD_ID}-{PUERTO}.proxy.runpod.net' if POD_ID else f'http://localhost:{PUERTO}'
print('INFERENCE_URL:', URL_PUBLICA)
print('INFERENCE_API_TOKEN:', API_TOKEN)

In [ ]:
# 2. Instalación de dependencias y Ollama con Qwen3:1.7b
!pip install -q ultralytics==8.3.27 easyocr==1.7.2 fastapi==0.115.4 "uvicorn[standard]==0.32.0" python-multipart opencv-python-headless requests
!pip install -q "git+https://github.com/ultralytics/CLIP.git"
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://localhost:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull $OLLAMA_MODEL

In [ ]:
# 3. Diagnóstico del entorno (Bonificación +5 pts)
from diagnosticar_entorno import diagnosticar
diagnosticar()

In [ ]:
# 4. Iniciar el servidor FastAPI en el puerto 8000
import threading, time, uvicorn
from app_inferencia import app, obtener_motor

motor = obtener_motor()
config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='info')
servidor = uvicorn.Server(config)
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(2)
print('Servidor iniciado en:', URL_PUBLICA)